# MiniMax-H3（现代 Web 音画视频工作台 + 全参数 API + Turbo 极速加速）

- **支持显卡**：80GB 及以上显存显卡（`A100 80GB` / `H100 80GB` / `RTX 6000 96GB`）
- **单页全能融合**：音画同生文生视频（`t2v`）、单首帧或首尾帧图生视频（`fl2v`）、多模态参考生视频（`r2v`，自由组合 `<Picture 1..5>` / `<Video 1..2>` / `<Audio 1..4>`），无需导入或切换 `.json` 工作流
- **Turbo 极速与热加载**：内置官方 **FL2V 8 步 Turbo** 与 **Ref2V 4 步 Turbo** 一键切换，并支持从 Hugging Face 或 Civitai / HTTPS 在线热挂载自定义 `.safetensors` LoRA
- **访问方式**：默认使用 Cloudflare Quick Tunnel 临时公网域名（用完即关）；亦可在第 4 步切换「自有 Tunnel」从 Colab Secrets 读取 `CF_TUNNEL_TOKEN` 挂载固定域名
- **执行顺序**：`1. 环境检测` → `2. 安装内部推理引擎与应用服务包` → `3. 下载并校验基础模型权重` → `4. 启动现代 Web 工作台与 API 服务`

| 入口路径 | 说明 |
| --- | --- |
| `GET /` | 现代亮色音画视频创作工作台（含独立分辨率/比例/17k+5 时长对齐、多模态标签插入、成片与参考图对比及会话历史） |
| `POST /api/v1/generate` | 异步提交文生音画视频 / 首尾帧视频 / 多模态参考视频生成任务（返回 `task_id`） |
| `POST /api/v1/files` | 上传首尾帧图片、多模态参考图片、参考视频或参考音频 |
| `GET /api/v1/tasks/{task_id}` | 查询实时推理步数进度与输出 MP4/WebM/MKV 地址（亦支持 `WS /api/v1/tasks/{task_id}/ws`） |
| `GET /docs` | Swagger / OpenAPI 在线交互接口文档 |

### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存、磁盘空间及 PyTorch / CUDA 版本。
- **要求**：支持 `A100 80GB`、`H100 80GB`、`RTX 6000 96GB`，显存 80GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 70
REQUIREMENT = "80GB 级显存（A100 80GB / H100 80GB / RTX 6000 96GB）"

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU。请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> A100 / H100 / RTX 6000 GPU。")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | 显存: {vram_gib:.1f} GiB | 系统内存: {ram_gib:.1f} GiB | 磁盘可用: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"当前显卡为 {gpu} ({vram_gib:.1f} GiB)，本笔记本需要 {REQUIREMENT}，请切换 runtime。")
if not bf16:
    raise RuntimeError("当前 GPU 不支持 BF16，请切换 runtime。")
print(f"[OK] GPU 环境正常：{gpu} ({vram_gib:.1f} GiB)")

### 2. 安装内部推理引擎与应用服务包
- **功能**：安装内部无头 GPU 推理算子引擎（ComfyUI `v0.43.2`）、FastAPI 服务依赖、`cloudflared` `2026.10.6`，并拉取校验 GitHub Release 发布的 `app-minimax-h3.tar.gz`（`v0.1.0`，内含编译好的现代前端 `dist` 与全参数 API 服务）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 2. 安装推理引擎与应用服务包 {display-mode: "form"}
import hashlib, re, shutil, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI"
COMFY_VERSION = "v0.43.2"
COMFY_REVISION = "423cbd20511becb1f16122ab518e92e58bfa03d9"

APP_DIR = Path("/content/app")
RELEASE_REPO = "boltguo/colab"
RELEASE_TAG = "v0.1.0"
APP_ARCHIVE = "app-minimax-h3.tar.gz"
APP_SHA256 = "d5b3bbfed438b9eaa748c210c43246ee019d53461f11c4275291c1090caad476"
CLOUDFLARED_VERSION = "2026.10.6"

EXTRA_PACKAGES = ["av>=19.0.1"]
APP_SERVER_PACKAGES = [
    "fastapi>=0.115.0,<1.0.0",
    "uvicorn[standard]>=0.30.0,<1.0.0",
    "python-multipart>=0.0.9,<1.0.0",
    "pydantic>=2.7.0,<3.0.0",
    "huggingface_hub>=0.23.0",
    "hf_xet",
]

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def git(path, *args):
    return subprocess.check_output(["git", *args], cwd=path, text=True).strip()

def ensure_not_symlink(path, label):
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"{label} 路径包含软链接，请使用新的 runtime：{p}")

def checkout_repo(path, repo_url, revision, label, allow_local_changes=False):
    ensure_not_symlink(path, label)
    fresh = not path.exists()
    if fresh:
        path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "--no-checkout", repo_url + ".git", str(path)])
    if not (path / ".git").is_dir():
        raise RuntimeError(f"{label} 目录不是 Git 仓库，请使用新的 runtime。")
    if git(path, "remote", "get-url", "origin").rstrip("/").removesuffix(".git") != repo_url:
        raise RuntimeError(f"现有 {label} 不是指定仓库，请使用新的 runtime。")
    if not fresh and not allow_local_changes and git(path, "status", "--porcelain"):
        raise RuntimeError(f"现有 {label} 有未提交修改，请使用新的 runtime。")
    if fresh or git(path, "rev-parse", "HEAD") != revision:
        run(["git", "fetch", "--depth", "1", "origin", revision], cwd=path)
        run(["git", "checkout", "--force", "--detach", revision], cwd=path)

def install_release_app():
    ensure_not_symlink(APP_DIR, "应用目录")
    stamp_file = APP_DIR / ".release_sha256"
    if (
        stamp_file.is_file()
        and not stamp_file.is_symlink()
        and stamp_file.read_text(encoding="utf-8").strip() == f"{RELEASE_TAG}:{APP_SHA256}"
        and (APP_DIR / "backend/run_server.py").is_file()
    ):
        print(f"应用包已是最新：{APP_ARCHIVE} ({RELEASE_TAG})")
        return

    local_archive = Path(f"/content/{APP_ARCHIVE}")
    with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        if local_archive.is_file() and not local_archive.is_symlink():
            print(f"检测到本地预置应用包：{local_archive}")
            payload = local_archive.read_bytes()
        else:
            url = f"https://github.com/{RELEASE_REPO}/releases/download/{RELEASE_TAG}/{APP_ARCHIVE}"
            print(f"正在拉取应用包：{url}")
            req = urllib.request.Request(url, headers={"User-Agent": "colab-release-fetcher"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                payload = resp.read()
        actual_sha = hashlib.sha256(payload).hexdigest()
        if actual_sha != APP_SHA256:
            raise RuntimeError(
                f"应用包 SHA256 校验失败！预期 {APP_SHA256}，实际 {actual_sha}"
            )
        tmp_path.write_bytes(payload)

        staging = APP_DIR.with_name("app.staging")
        if staging.is_symlink():
            raise RuntimeError("staging 目录为软链接，拒绝写入。")
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)
        staging_root = staging.resolve()

        with tarfile.open(tmp_path, "r:gz") as tf:
            for member in tf.getmembers():
                if member.issym() or member.islnk():
                    raise RuntimeError(f"拒绝解压含链接的归档条目：{member.name}")
                target = (staging / member.name).resolve()
                if target != staging_root and not str(target).startswith(str(staging_root) + "/"):
                    raise RuntimeError(f"归档路径越界：{member.name}")
            tf.extractall(staging)

        (staging / ".release_sha256").write_text(f"{RELEASE_TAG}:{APP_SHA256}\n", encoding="utf-8")
        if APP_DIR.exists():
            shutil.rmtree(APP_DIR)
        staging.replace(APP_DIR)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

checkout_repo(COMFY_DIR, COMFY_REPO, COMFY_REVISION, "ComfyUI")
REQUIREMENT_FILES = [COMFY_DIR / "requirements.txt"]

skip = {"torch", "torchvision", "torchaudio", "xformers"}
lines = []
for req_file in REQUIREMENT_FILES:
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        name = re.split(r"[<>=!~\[;\s]", s, maxsplit=1)[0].strip().lower()
        if name not in skip:
            lines.append(s)
requirements = Path("/content/comfy_requirements.txt")
requirements.write_text("\n".join(lines) + "\n", encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), *EXTRA_PACKAGES, *APP_SERVER_PACKAGES])

install_release_app()

cloudflared_bin = Path("/usr/local/bin/cloudflared")
try:
    installed = subprocess.run([str(cloudflared_bin), "--version"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
except OSError:
    installed = ""
if CLOUDFLARED_VERSION not in installed.split():
    download = cloudflared_bin.with_name("cloudflared.download")
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64",
        str(download),
    )
    download.chmod(0o755)
    download.replace(cloudflared_bin)

print(f"Engine: ComfyUI {COMFY_VERSION} ({COMFY_REVISION[:12]}) | app: {RELEASE_TAG} ({APP_SHA256[:12]}) | cloudflared: {CLOUDFLARED_VERSION}")
print("[OK] 内部推理引擎与现代 Web/API 应用包安装完成。")

### 3. 下载并校验基础模型权重
- **功能**：高速下载并精确校验 MiniMax H3 FL2VA / Ref2VA 音画同生主模型、Qwen3-VL-32B 编码器、双模态 VAE 与官方 8/4 步 Turbo LoRA（约 71.2 GB）至 `/content/ComfyUI/models`（自定义风格或角色 `.safetensors` LoRA 无需在此预先全量下载，启动后可在 Web 工作台或 API 中按需在线热挂载）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 3. 下载并校验基础模型权重 {display-mode: "form"}
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import hf_hub_download

COMFY_DIR = Path("/content/ComfyUI")
MODEL_DIR = COMFY_DIR / "models"
H3 = ("Comfy-Org/MiniMax-H3", "e5eb578a89295337b8ff433a035929ce0279e0b6", "")
TURBO = ("lightx2v/Minimax-h3-Turbo", "30f485ed874aa44c8183348ad361c5eabad318da", "loras")

MODEL_FILES = [
    (*H3, "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", 20970379616),
    (*H3, "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors", 20970379616),
    (*H3, "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors", 27141342152),
    (*H3, "vae/minimax_h3_video_vae_int8_convrot.safetensors", 2811065184),
    (*H3, "vae/minimax_h3_audio_vae_fp32.safetensors", 605254808),
    (*H3, "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", 1956193000),
    (*TURBO, "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", 1956193000),
]

def local_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == COMFY_DIR.parent:
            break
        if parent.is_symlink():
            raise RuntimeError(f"目录是软链接，请使用新的 runtime：{parent}")
    path.mkdir(parents=True, exist_ok=True)

def model_ready(target, size):
    return target.is_file() and not target.is_symlink() and target.stat().st_size == size

def download_model(repo, revision, folder, file, size):
    base = MODEL_DIR / folder if folder else MODEL_DIR
    target = base / file
    local_directory(target.parent)
    started = time.time()
    result = Path(
        hf_hub_download(
            repo_id=repo,
            filename=file,
            revision=revision,
            local_dir=str(base),
            force_download=target.exists(),
            token=False,
        )
    )
    if result.resolve() != target.resolve() or not model_ready(target, size):
        raise RuntimeError(f"模型下载不完整或校验不匹配：{target}")
    print(f"已下载：{file}（{size / 1024**3:.2f} GB，{time.time() - started:.0f} 秒）")

def download_models():
    pending = []
    for item in MODEL_FILES:
        base = MODEL_DIR / item[2] if item[2] else MODEL_DIR
        target = base / item[3]
        if target.is_symlink():
            raise RuntimeError(f"模型文件是软链接：{target}")
        if model_ready(target, item[4]):
            print("已就绪：", item[3])
        else:
            pending.append(item)
    if not pending:
        return
    missing = sum(item[4] for item in pending)
    free = shutil.disk_usage(MODEL_DIR).free
    if free < missing + 2 * 1024**3:
        raise RuntimeError(f"磁盘空间不足：还需 {missing / 1024**3:.1f} GB，当前可用 {free / 1024**3:.1f} GB。")
    started = time.time()
    pool = ThreadPoolExecutor(max_workers=3)
    try:
        futures = [pool.submit(download_model, *item) for item in sorted(pending, key=lambda item: -item[4])]
        for future in as_completed(futures):
            future.result()
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    elapsed = time.time() - started
    print(f"基础模型下载完成：{missing / 1024**3:.1f} GB，用时 {elapsed:.0f} 秒（{missing / 1024**2 / max(elapsed, 1):.0f} MB/s）")

for path in [MODEL_DIR, MODEL_DIR / "loras", COMFY_DIR / "input", COMFY_DIR / "output"]:
    local_directory(path)
download_models()
print("[OK] 基础模型权重校验完成（可选风格与自定义 LoRA 可在 Web 工作台或 API 中按需在线热挂载）。")

### 4. 启动现代 Web 工作台与 API 服务
- **功能**：启动 FastAPI 服务（`127.0.0.1:8188`，同时承载 `/` 现代前端工作台、`/api/v1/*` 业务接口与 `/docs` OpenAPI 文档），并挂载 Cloudflare Tunnel。
- **隧道选择**：`TUNNEL_MODE` 默认选择 `Quick Tunnel (临时链接)`（免费公网临时域名，实例关闭即失效）；也可切换为 `自有 Tunnel`，从 Colab 左侧 **🔑 Secrets** 读取 `CF_TUNNEL_TOKEN` 挂载固定域名。
- **接口鉴权**：可选勾选 `REQUIRE_API_KEY`，从 Colab Secrets 读取 `APP_API_KEY` 开启 Bearer Token 鉴权（开启后在网页右上角「API Key」填入同一密钥即可）。
- **使用**：直接运行，点击输出的公网链接即可在网页操作或直接复制对应链接调用 API。

In [ ]:
# @title 4. 启动现代 Web 工作台与 API 服务 {display-mode: "form"}
TUNNEL_MODE = "Quick Tunnel (临时链接)"  # @param ["Quick Tunnel (临时链接)", "自有 Tunnel (读取 Secrets 中 CF_TUNNEL_TOKEN)"]
REQUIRE_API_KEY = False  # @param {type:"boolean"}
USE_CF_TUNNEL_TOKEN = TUNNEL_MODE.startswith("自有")
import hashlib, json, os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

FEATURE_NAME = "minimax-h3"
COMFY_DIR = Path("/content/ComfyUI")
COMFY_PORT = 8189
COMFY_BASE = f"http://127.0.0.1:{COMFY_PORT}"
COMFY_LOG = Path("/content/comfyui-worker.log")
COMFY_LAUNCHER = Path("/content/comfy_userdata_compat.py")
COMFY_LAUNCH_CONFIG = Path("/content/comfyui-worker.launch.json")
COMFY_PROCESS_PATTERN = r"comfy_userdata_compat\.py|ComfyUI/main\.py"
COMFY_LAUNCH_ARGS = [
    "--output-directory", str(COMFY_DIR / "output"),
    "--disable-all-custom-nodes",
    "--vram-headroom", "4",
    "--use-pytorch-cross-attention",
    "--disable-partner-nodes",
]
REQUIRED_NODES = {"MiniMaxH3ImageToVideo", "MiniMaxH3ReferenceToVideo", "MiniMaxH3AddGuide", "MiniMaxH3SigmaShift", "VAEDecodeAudio", "CreateVideo", "SaveVideo"}

APP_DIR = Path("/content/app")
DATA_DIR = Path(f"/content/app_data/{FEATURE_NAME}")
PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
APP_LOG = Path(f"/content/{FEATURE_NAME}-web.log")
APP_LAUNCH_CONFIG = Path(f"/content/{FEATURE_NAME}.launch.json")
APP_PROCESS_PATTERN = r"backend/run_server\.py|backend\.run_server"

CF_BIN = Path("/usr/local/bin/cloudflared")
CF_LOG = Path("/content/cloudflared.log")

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token=""):
    if not CF_BIN.exists():
        raise RuntimeError("未找到 cloudflared，请先运行第 2 步。")
    stop_processes("cloudflared tunnel", 10)
    rotate_log(CF_LOG)
    cmd = [
        str(CF_BIN),
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--edge-ip-version", "4",
    ]
    if token:
        cmd.extend(["run", "--url", BASE, "--token", token])
    else:
        cmd.extend(["--url", BASE])
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            cmd,
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    deadline = time.time() + (30 if token else 60)
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(tail(CF_LOG, 4000))
            if token:
                raise RuntimeError("Cloudflare Tunnel 启动失败，请检查 CF_TUNNEL_TOKEN 是否正确。")
            raise RuntimeError("Cloudflare 临时隧道启动失败，请查看上方日志。")
        log_text = read_text(CF_LOG)
        if not token:
            match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
            if match and "Registered tunnel connection" in log_text:
                url = match.group(0)
                print(f"[OK] Cloudflare 临时隧道已连接 (HTTP/2): {url} -> {BASE}")
                return url
        elif "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = read_text(CF_LOG)
            configs = [line for line in log_text.splitlines() if "Updated to new configuration" in line]
            config = configs[-1].replace('\\"', '"') if configs else ""
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"(?:(?!"hostname").)*?"service"\s*:\s*"([^"]+)"', config)
            for host, service in routes:
                print(f"[OK] Cloudflare Tunnel 已连接 (HTTP/2): https://{host} -> {service}")
            hosts = [host for host, service in routes if f":{PORT}" in service]
            if not routes:
                print("[OK] Cloudflare Tunnel 已连接 (HTTP/2)，请通过你在 Cloudflare 绑定的域名访问。")
            elif not hosts:
                print(f"[WARN] 没有域名转发到本服务的 {PORT} 端口！")
                print(f"       请前往 Cloudflare Zero Trust -> Networks -> Tunnels -> Public Hostname，将 Service URL 改为 {BASE}")
            return f"https://{hosts[0]}" if hosts else "https://<你的CF域名>"
        time.sleep(1)
    if not token:
        match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", read_text(CF_LOG))
        if match:
            url = match.group(0)
            print(f"[OK] Cloudflare 临时隧道已创建: {url} -> {BASE}")
            return url
        print(tail(CF_LOG, 4000))
        raise RuntimeError(f"Cloudflare 临时隧道 60 秒内未返回地址，详见 {CF_LOG}。")
    print(f"[WARN] Cloudflare Tunnel 30 秒内未确认连接，仍在后台运行，详见 {CF_LOG}。")
    return "https://<你的CF域名>"

def check_public_url(url, route):
    if "<" in url:
        return
    error = None
    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            request = urllib.request.Request(url + route, headers={"User-Agent": "colab-tunnel-check"})
            with urllib.request.urlopen(request, timeout=10):
                print(f"[OK] 公网访问正常：{url}")
                return
        except Exception as e:
            error = e
        time.sleep(3)
    print(f"[WARN] 公网地址 60 秒内无法访问：{url}{route}（{error}）")

cf_token = resolve_cf_token() if USE_CF_TUNNEL_TOKEN else ""
if USE_CF_TUNNEL_TOKEN and not cf_token:
    raise RuntimeError("已勾选 USE_CF_TUNNEL_TOKEN，但未检测到 CF_TUNNEL_TOKEN；请在 Colab 左侧 🔑 Secrets 中添加 CF_TUNNEL_TOKEN 并开启访问权限，或取消勾选改用临时隧道。")

if not (COMFY_DIR / "main.py").is_file():
    raise RuntimeError("未找到内部推理引擎 /content/ComfyUI，请先运行第 2 步。")
if APP_DIR.is_symlink() or not (APP_DIR / "backend/run_server.py").is_file():
    raise RuntimeError("未找到应用服务目录 /content/app，请先运行第 2 步。")

if FEATURE_NAME in ("minimax-h3", "seedvr2"):
    missing_media_tools = [name for name in ("ffmpeg", "ffprobe") if not shutil.which(name)]
    if missing_media_tools:
        raise RuntimeError(f"缺少媒体处理工具：{', '.join(missing_media_tools)}。请先执行 !apt-get -qq update && apt-get -qq install -y ffmpeg，再运行第 4 步。")

def resolve_app_api_key():
    key = ""
    try:
        from google.colab import userdata
        key = (userdata.get("APP_API_KEY") or "").strip()
    except Exception:
        key = ""
    if not key:
        key = os.environ.get("APP_API_KEY", "").strip()
    return key

app_api_key = resolve_app_api_key() if REQUIRE_API_KEY else ""
if REQUIRE_API_KEY and not app_api_key:
    raise RuntimeError(
        "已勾选 REQUIRE_API_KEY，但未找到 APP_API_KEY。请在 Colab 左侧 🔑 Secrets 中配置 APP_API_KEY。"
    )

LAUNCHER_SOURCE = r'''
"""Start pinned ComfyUI with narrow workflow aliases and Cloudflare compatibility."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI 工作流访问接口已变化，未安装兼容路由。")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def install_cf_middleware():
    import middleware.cache_middleware as cm
    orig_cache_control = cm.cache_control

    @web.middleware
    async def cf_safe_middleware(request, handler):
        response = await orig_cache_control(request, handler)
        if request.method == "GET" and request.path == "/" and getattr(response, "status", None) == 200:
            import server
            inst = getattr(server.PromptServer, "instance", None)
            web_root = getattr(inst, "web_root", None) if inst else None
            index_file = Path(web_root) / "index.html" if web_root else None
            if index_file and index_file.is_file():
                html = index_file.read_text(encoding="utf-8", errors="replace")
                html = html.replace("<script ", '<script data-cfasync="false" ')
                new_resp = web.Response(text=html, content_type="text/html")
                for k, v in response.headers.items():
                    if k.lower() not in {"content-length", "content-type", "etag", "last-modified"}:
                        new_resp.headers[k] = v
                new_resp.headers["Cache-Control"] = "no-store, no-transform, must-revalidate"
                response = new_resp
        if isinstance(response, web.StreamResponse) and not isinstance(response, web.WebSocketResponse) and not response.prepared:
            if getattr(response, "status", None) == 200 and (
                request.path.startswith("/assets/")
                or request.path.endswith((".woff2", ".woff", ".ttf"))
            ):
                response.headers["Cache-Control"] = "public, max-age=604800, immutable, no-transform"
            else:
                cc = response.headers.get("Cache-Control", "")
                if "no-transform" not in cc:
                    response.headers["Cache-Control"] = f"{cc}, no-transform" if cc else "no-transform"
            origin = request.headers.get("Origin")
            if origin and response.headers.get("Access-Control-Allow-Origin") == "*":
                response.headers["Access-Control-Allow-Origin"] = origin
                response.headers["Vary"] = "Origin"
        return response

    cm.cache_control = cf_safe_middleware


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("ComfyUI 源码目录缺失或无效。")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    install_cf_middleware()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''

def comfy_get_json(route, timeout=5):
    with urllib.request.urlopen(COMFY_BASE + route, timeout=timeout) as response:
        return json.load(response)

def comfy_ready(timeout=5):
    try:
        comfy_get_json("/system_stats", timeout)
        return True
    except Exception:
        return False

# 1. 启动内部无头推理 Worker (127.0.0.1:8189，不对公网直接暴露)
comfy_cfg_str = json.dumps({"launcher": LAUNCHER_SOURCE, "args": COMFY_LAUNCH_ARGS, "port": COMFY_PORT})
comfy_running = comfy_ready() or (process_running(COMFY_PROCESS_PATTERN) and comfy_ready(timeout=30))
if comfy_running and read_text(COMFY_LAUNCH_CONFIG) != comfy_cfg_str:
    print("内部推理引擎参数已更新，正在重启...")
    stop_processes(COMFY_PROCESS_PATTERN, 60)
    comfy_running = False

if comfy_running:
    print("[OK] 内部推理引擎已在运行 (127.0.0.1:8189)。")
else:
    stop_processes(COMFY_PROCESS_PATTERN, 60)
    COMFY_LAUNCHER.write_text(LAUNCHER_SOURCE, encoding="utf-8")
    rotate_log(COMFY_LOG)
    with COMFY_LOG.open("w") as log:
        comfy_proc = subprocess.Popen(
            [sys.executable, str(COMFY_LAUNCHER), "--listen", "127.0.0.1", "--port", str(COMFY_PORT), *COMFY_LAUNCH_ARGS],
            cwd=COMFY_DIR,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    COMFY_LAUNCH_CONFIG.write_text(comfy_cfg_str, encoding="utf-8")
    print("正在启动内部 GPU 推理引擎...")
    deadline = time.time() + 600
    while time.time() < deadline:
        if comfy_proc.poll() is not None:
            print(tail(COMFY_LOG))
            raise RuntimeError(f"内部推理引擎启动失败（退出码 {comfy_proc.returncode}），详见上方日志。")
        if comfy_ready():
            print("[OK] 内部 GPU 推理引擎就绪 (127.0.0.1:8189)。")
            break
        time.sleep(2)
    else:
        print(tail(COMFY_LOG))
        raise TimeoutError("内部推理引擎 10 分钟内未就绪，请检查日志。")

missing = sorted(REQUIRED_NODES - set(comfy_get_json("/object_info", timeout=60)))
if missing:
    print(tail(COMFY_LOG))
    raise RuntimeError(f"推理引擎缺少必要算子节点：{', '.join(missing)}。请重新运行第 2 步。")

# 2. 启动对外 FastAPI + 现代静态前端工作台 (127.0.0.1:8188)
def app_ready(timeout=5):
    try:
        with urllib.request.urlopen(BASE + "/api/v1/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

env_overrides = {
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "COLAB_FEATURE": FEATURE_NAME,
    "COLAB_HOST": "127.0.0.1",
    "COLAB_PORT": str(PORT),
    "COLAB_DATA_DIR": str(DATA_DIR),
    "COLAB_DIST_DIR": str(APP_DIR / "dist" / FEATURE_NAME),
    "COLAB_API_KEY": app_api_key,
    "COMFY_BASE_URL": COMFY_BASE,
    "COMFY_DIR": str(COMFY_DIR),
}

app_cmd = [
    sys.executable,
    str(APP_DIR / "backend/run_server.py"),
    "--feature",
    FEATURE_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--dist-dir",
    str(APP_DIR / "dist" / FEATURE_NAME),
]

app_fingerprint = json.dumps({
    "cmd": app_cmd,
    "release": read_text(APP_DIR / ".release_sha256").strip(),
    "auth_sha256": hashlib.sha256(app_api_key.encode("utf-8")).hexdigest(),
    "env": {key: value for key, value in env_overrides.items() if key != "COLAB_API_KEY"},
}, sort_keys=True)
app_is_up = app_ready() or (process_running(APP_PROCESS_PATTERN) and app_ready(timeout=15))
if app_is_up and read_text(APP_LAUNCH_CONFIG) != app_fingerprint:
    print("Web/API 服务配置变更，正在重启...")
    stop_processes(APP_PROCESS_PATTERN, 30)
    app_is_up = False

if app_is_up:
    print("[OK] 现代 Web 工作台与 API 服务已在运行 (127.0.0.1:8188)。")
else:
    stop_processes(APP_PROCESS_PATTERN, 30)
    rotate_log(APP_LOG)
    child_env = {**os.environ, **env_overrides}
    with APP_LOG.open("w") as log:
        app_proc = subprocess.Popen(
            app_cmd,
            cwd=APP_DIR,
            env=child_env,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    APP_LAUNCH_CONFIG.write_text(app_fingerprint, encoding="utf-8")
    print("正在启动现代 Web 工作台与标准 API 网关...")
    deadline = time.time() + 120
    while time.time() < deadline:
        if app_proc.poll() is not None:
            print(tail(APP_LOG))
            raise RuntimeError(f"Web/API 服务启动失败（退出码 {app_proc.returncode}），详见上方日志。")
        if app_ready():
            print("[OK] 现代 Web 工作台与标准 API 服务启动成功。")
            break
        time.sleep(1.5)
    else:
        print(tail(APP_LOG))
        raise TimeoutError("Web/API 服务 2 分钟内未就绪，请检查日志。")

# 3. 挂载 Cloudflare Tunnel 并同步隧道地址供前端 API 调用示例自动读取
host_url = start_cf_tunnel(cf_token)
tunnel_mode_key = "custom" if cf_token else "quick"
if not DATA_DIR.is_symlink():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    tunnel_info_path = DATA_DIR / "tunnel_info.json"
    if not tunnel_info_path.is_symlink():
        tunnel_info_path.write_text(
            json.dumps({"public_url": host_url, "tunnel_mode": tunnel_mode_key}, ensure_ascii=False),
            encoding="utf-8",
        )
check_public_url(host_url, "/api/v1/health")

sample_body = json.dumps({'mode': 't2v', 'prompt': '清晨阳光洒在海面，海浪轻拍礁石，远处海鸥鸣叫，电影级推镜头，自然立体声海浪音效', 'resolution_tier': '768p', 'aspect_ratio': '16:9', 'duration_sec': 5.0, 'turbo_enabled': True, 'steps': 8}, ensure_ascii=False)
auth_note = ' -H "Authorization: Bearer $APP_API_KEY"' if app_api_key else ""
tunnel_tip = (
    "⚠️ 提示：当前使用 Cloudflare Quick Tunnel 临时公网域名，实例关闭或重连隧道后地址即失效；如需固定域名请在上方选择「自有 Tunnel」。"
    if tunnel_mode_key == "quick"
    else "✅ 当前使用自有 Cloudflare Tunnel 固定域名。"
)
print(f"""
=== 访问入口 ===
🌐 现代 Web 操作台 : {host_url}
📖 在线 API 文档   : {host_url}/docs
🩺 机器与显存状态  : {host_url}/api/v1/system
{tunnel_tip}

=== 快速 API 调用示例 ===
curl -X POST "{host_url}/api/v1/generate" \\
  -H "Content-Type: application/json"{auth_note} \\
  -d '{sample_body}'
""")